# BraTS 2024 - Slice Dataset Generation and Export (Local Workstation)

This notebook handles:
1. Environment setup and repository path initialization.
2. Loading preprocessing configuration and patient splits (`train` / `validate` / `test`).
3. **Single patient sanity check:** validating slice extraction, PyTorch tensors, and visualization.
4. **Full dataset export:** extracting and saving all 2D `.pt` slices to your local SSD.

## 1. Environment and repository setup

In [ ]:
import sys
from pathlib import Path

# Automatically resolve repository root (whether running from notebooks/ or project root)
repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
repo_src = repo_root / "src"

if str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

import torch
import numpy as np

from brain_mri.config import load_data_config
from brain_mri.data.split import get_or_create_splits
from brain_mri.data.slice_extraction import extract_patient_slices
from brain_mri.data.dataset_export import save_patients_to_pt, export_dataset_split

print(f"Repository root: {repo_root}")
print(f"PyTorch version: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")

## 2. Data paths and configuration

In [ ]:
# Set the path to where your raw BraTS dataset is extracted on your local drive:
# e.g. Path("D:/BraTS/training_data1_v2") or repo_root / "data" / "raw" / "training_data1_v2"
RAW_DATA_ROOT = Path("D:/BraTS/training_data1_v2")

# Destination directory for preprocessed 2D .pt slices (best placed on an SSD):
PROCESSED_ROOT = repo_root / "data" / "processed"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

# Load preprocessing YAML configuration
config = load_data_config(repo_root / "configs" / "data_config.yaml")
config.paths.z_boundaries = repo_root / "data" / "patients_z_boundaries.json"

# Load or generate patient splits (train: 70%, val: 15%, test: 15%)
splits_json_path = repo_root / "data" / "data_splits.json"
splits = get_or_create_splits(folder_path=RAW_DATA_ROOT, json_path=splits_json_path)

print(f"Raw data path:       {RAW_DATA_ROOT} (exists: {RAW_DATA_ROOT.exists()})")
print(f"Processed destination: {PROCESSED_ROOT}")
print(f"Patient count:")
print(f" - Train:      {len(splits['train'])}")
print(f" - Validation: {len(splits['validate'])}")
print(f" - Test:       {len(splits['test'])}")

## 3. Single patient test (Sanity check and visualization)

In [ ]:
import matplotlib.pyplot as plt

# Select the first training patient for testing
test_patient_id = splits["train"][0]
test_patient_dir = RAW_DATA_ROOT / test_patient_id
print(f"Testing patient: {test_patient_id}")

# Extract slices
slices = extract_patient_slices(test_patient_dir, config=config)
print(f"Extracted {len(slices)} slices.")

# Save slices to a temporary test directory
test_output_dir = repo_root / "data" / "test_processed"
saved_files = save_patients_to_pt(slices, test_output_dir)
print(f"Saved {len(saved_files)} .pt files to {test_output_dir}")

# Inspect a middle slice (where tumor is usually present)
sample_file = saved_files[len(saved_files) // 2]
data = torch.load(sample_file, weights_only=True)

img = data["image"]  # (4, 224, 224)
mask = data["mask"]  # (224, 224)

print("\n--- Tensor verification ---")
print(f"Image shape:       {img.shape}, dtype: {img.dtype}")
print(f"Mask shape:        {mask.shape}, dtype: {mask.dtype}")
print(f"Unique mask labels: {torch.unique(mask).tolist()}")
print(f"Background value (corner): {img[:, 0, 0].tolist()}")

# Plot all 4 MRI modalities + segmentation mask
modalities = ["T1", "T1CE", "T2", "FLAIR"]
fig, axes = plt.subplots(1, 5, figsize=(18, 4))

for i in range(4):
    axes[i].imshow(img[i].numpy(), cmap="gray")
    axes[i].set_title(modalities[i])
    axes[i].axis("off")

axes[4].imshow(mask.numpy(), cmap="tab10", vmin=0, vmax=4)
axes[4].set_title(f"Mask (z={data['z_idx']})")
axes[4].axis("off")

plt.tight_layout()
plt.show()

## 4. Full dataset export (to local SSD)

Extracts all slices across train, validation, and test splits.
Automatic resume is enabled: if interrupted, re-running this cell skips already completed patients.

In [ ]:
stats = export_dataset_split(
    data_root=RAW_DATA_ROOT,
    splits=splits,
    output_root=PROCESSED_ROOT,
    config=config,
    overwrite=False,  # Automatically resumes if previously interrupted
)
print("\nExport summary:", stats)